# Day 13 — Window Functions

**What you will learn:**
- `Window` spec: `partitionBy`, `orderBy`, `rowsBetween`, `rangeBetween`
- Ranking: `rank`, `dense_rank`, `row_number`, `percent_rank`, `ntile`
- Navigation: `lag`, `lead`, `first`, `last`
- Aggregation over a window: running totals, moving averages

**Exam relevance:** DataFrame API (30%) — window functions are heavily tested. Know the difference between `rank` and `dense_rank`, and how frame boundaries work.

In [ ]:
from pyspark.sql.functions import (
    col, rank, dense_rank, row_number, percent_rank, ntile,
    lag, lead, first, last,
    sum, avg, min, max, count, round
)
from pyspark.sql.window import Window

data = [
    ("Alice",   "Engineering", 2022, 95000),
    ("Bob",     "Marketing",   2021, 72000),
    ("Carol",   "Engineering", 2020, 110000),
    ("Dave",    "Sales",       2023, 58000),
    ("Eve",     "Marketing",   2021, 81000),
    ("Frank",   "Engineering", 2022, 88000),
    ("Grace",   "Sales",       2020, 61000),
    ("Hank",    "Engineering", 2023, 102000),
    ("Iris",    "Marketing",   2022, 76000),
    ("Jack",    "Sales",       2021, 67000),
]
df = spark.createDataFrame(data, ["name", "dept", "hire_year", "salary"])
df.show()

## 1. Window Spec — the Building Block

A `Window` spec defines **which rows** are visible to the function:

```
Window.partitionBy(...)   # reset the window per group (like GROUP BY)
      .orderBy(...)       # defines row order within the partition
      .rowsBetween(...)   # physical row offsets
      .rangeBetween(...)  # value-based offsets
```

In [ ]:
# Partition by dept, order by salary descending
w = Window.partitionBy("dept").orderBy(col("salary").desc())

df.withColumn("rank",       rank().over(w)) \
  .withColumn("dense_rank", dense_rank().over(w)) \
  .withColumn("row_num",    row_number().over(w)) \
  .select("dept", "name", "salary", "rank", "dense_rank", "row_num") \
  .orderBy("dept", "rank") \
  .show()

## 2. rank vs dense_rank vs row_number

| Function | Ties | Gap after tie | Deterministic? |
|---|---|---|---|
| `rank` | Same rank | Yes (skips numbers) | No for ties |
| `dense_rank` | Same rank | No gap | No for ties |
| `row_number` | Unique per row | No gap | No for ties — arbitrary tiebreak |

> **Exam tip:** If two rows have the same salary, `rank` gives both rank 1 and skips rank 2. `dense_rank` gives both rank 1 and the next distinct salary gets rank 2.

In [ ]:
# Demonstrate tie behaviour — same salary for Alice and Frank (both Engineering)
df_tie = spark.createDataFrame([
    ("Alice",  "Eng", 95000),
    ("Frank",  "Eng", 95000),  # same salary = tie
    ("Hank",   "Eng", 102000),
    ("Carol",  "Eng", 110000),
], ["name", "dept", "salary"])

w2 = Window.partitionBy("dept").orderBy(col("salary").desc())

df_tie.withColumn("rank",       rank().over(w2)) \
      .withColumn("dense_rank", dense_rank().over(w2)) \
      .withColumn("row_number", row_number().over(w2)) \
      .show()

## 3. Top-N Per Group — Common Pattern

In [ ]:
# Top 2 earners per department
w_sal = Window.partitionBy("dept").orderBy(col("salary").desc())

df.withColumn("rnk", dense_rank().over(w_sal)) \
  .filter(col("rnk") <= 2) \
  .select("dept", "name", "salary", "rnk") \
  .orderBy("dept", "rnk") \
  .show()

## 4. percent_rank and ntile

In [ ]:
w_global = Window.orderBy(col("salary").desc())

df.withColumn("pct_rank",  round(percent_rank().over(w_global), 2)) \
  .withColumn("quartile",  ntile(4).over(w_global)) \
  .select("name", "salary", "pct_rank", "quartile") \
  .orderBy("salary", ascending=False) \
  .show()

## 5. lag and lead — Access Neighbouring Rows

- `lag(col, offset, default)` — value from N rows **before** the current row
- `lead(col, offset, default)` — value from N rows **after** the current row

Useful for: change detection, delta calculations, session analysis.

In [ ]:
# Year-over-year salary change per employee (using hire_year ordering)
sales = df.filter(col("dept") == "Sales").orderBy("hire_year")

w_yr = Window.partitionBy("dept").orderBy("hire_year")

sales.withColumn("prev_year_salary", lag("salary", 1).over(w_yr)) \
     .withColumn("next_year_salary", lead("salary", 1).over(w_yr)) \
     .select("name", "hire_year", "salary", "prev_year_salary", "next_year_salary") \
     .show()

# Delta: how salary changed since previous hire in same dept
df.withColumn("prev_sal", lag("salary", 1, 0).over(w_yr)) \
  .withColumn("delta",    col("salary") - col("prev_sal")) \
  .select("dept", "name", "hire_year", "salary", "delta") \
  .orderBy("dept", "hire_year") \
  .show()

## 6. Aggregate Window Functions — Running Totals & Moving Averages

Frame boundaries control **which rows** the aggregate sees:

```python
Window.rowsBetween(Window.unboundedPreceding, Window.currentRow)   # cumulative
Window.rowsBetween(-2, 0)                                           # 3-row rolling
Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)  # whole partition
```

In [ ]:
w_cum = Window.partitionBy("dept") \
              .orderBy("hire_year") \
              .rowsBetween(Window.unboundedPreceding, Window.currentRow)

w_roll = Window.partitionBy("dept") \
               .orderBy("hire_year") \
               .rowsBetween(-1, 1)   # current + 1 before + 1 after

w_whole = Window.partitionBy("dept")  # no orderBy → whole partition visible

df.withColumn("cumulative_sum",  sum("salary").over(w_cum)) \
  .withColumn("rolling_avg",     round(avg("salary").over(w_roll), 0)) \
  .withColumn("dept_avg",        round(avg("salary").over(w_whole), 0)) \
  .withColumn("dept_min",        min("salary").over(w_whole)) \
  .select("dept", "name", "hire_year", "salary",
          "cumulative_sum", "rolling_avg", "dept_avg", "dept_min") \
  .orderBy("dept", "hire_year") \
  .show()

## 7. first and last Over a Window

In [ ]:
w_full = Window.partitionBy("dept").orderBy("hire_year") \
               .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

df.withColumn("earliest_hire", first("name").over(w_full)) \
  .withColumn("latest_hire",   last("name").over(w_full)) \
  .select("dept", "name", "hire_year", "earliest_hire", "latest_hire") \
  .orderBy("dept", "hire_year") \
  .show()

---

## Window Functions Quick Reference

| Function | What it computes |
|---|---|
| `rank()` | Rank with gaps on ties |
| `dense_rank()` | Rank without gaps on ties |
| `row_number()` | Unique sequential number |
| `percent_rank()` | Relative rank 0.0–1.0 |
| `ntile(n)` | Bucket number (1..n) |
| `lag(col, n, default)` | Value n rows before |
| `lead(col, n, default)` | Value n rows after |
| `first(col)` | First value in frame |
| `last(col)` | Last value in frame |
| `sum/avg/min/max(col)` | Aggregate over frame |

## Day 13 Checklist

- [ ] Built a `Window` spec with `partitionBy` + `orderBy`
- [ ] Applied `rank`, `dense_rank`, `row_number` — know tie behaviour
- [ ] Extracted top-N per group using `dense_rank` + `.filter()`
- [ ] Used `lag` and `lead` for delta calculations
- [ ] Built cumulative sum using `rowsBetween(unboundedPreceding, currentRow)`
- [ ] Built rolling average using `rowsBetween(-N, N)`

**Next:** Day 14 — Caching & Persistence